# Рендер Blender в Google Colab

Этот ноутбук рендерит уже подготовленный проект Blender без изменения его сцены, камер, движка или настроек качества.

**Перед загрузкой:** если в сцене используются внешние текстуры, HDRI, кеши симуляций или другие файлы, либо упакуйте их в Blender через `File → External Data → Pack Resources`, либо загрузите ZIP-архив с папкой проекта. Внутри ZIP должен быть `.blend` и сохранена исходная структура папок.

Для Cycles включите в Colab: `Runtime → Change runtime type → T4 GPU` (или другой GPU).

In [ ]:
#@title 1. Проверенная конфигурация
# Выберите LTS-версию. `custom` разрешает только явную версию X.Y.Z.
BLENDER_VERSION_PRESET = '5.2.1' #@param ['5.2.1', '4.5.13', 'custom']
CUSTOM_BLENDER_VERSION = '' #@param {type: 'string'}

# True: выбрать GPU средствами Blender/Cycles.
ENABLE_CYCLES_GPU = True #@param {type: 'boolean'}
# Без явного True отсутствие GPU останавливает Cycles до дорогостоящего CPU-рендера.
ALLOW_CPU_FALLBACK = False #@param {type: 'boolean'}
# По умолчанию CPU выключается при выбранном GPU.
INCLUDE_CPU_WITH_GPU = False #@param {type: 'boolean'}

RENDER_MODE = 'ANIMATION' #@param ['ANIMATION', 'STILL']
STILL_FRAME = 1 #@param {type: 'integer'}
DOWNLOAD_RESULT = False #@param {type: 'boolean'}

# Запускает отдельный настоящий Cycles 64x64 GPU smoke test до загрузки проекта.
RUN_CYCLES_SMOKE_TEST = False #@param {type: 'boolean'}
# Рендерит один кадр после read-only preflight для оценки времени/места; не изменяет .blend.
RUN_PREFLIGHT_TEST_FRAME = False #@param {type: 'boolean'}

from dataclasses import dataclass
from enum import Enum
import re

LTS_VERSION_PRESETS = ('5.2.1', '4.5.13')
CUSTOM_VERSION_PRESET = 'custom'
VERSION_RE = re.compile(r'^(0|[1-9]\d*)\.(0|[1-9]\d*)\.(0|[1-9]\d*)$')

class ConfigurationError(ValueError):
    pass

class RenderMode(str, Enum):
    STILL = 'STILL'
    ANIMATION = 'ANIMATION'

def validate_blender_version(version):
    if not isinstance(version, str) or not VERSION_RE.fullmatch(version.strip()):
        raise ConfigurationError('Версия Blender должна быть точной строкой X.Y.Z, например 5.2.1.')
    return version.strip()

def resolve_blender_version(preset, custom_version):
    if not isinstance(preset, str) or not isinstance(custom_version, str):
        raise ConfigurationError('Значения версии должны быть строками.')
    preset = preset.strip()
    if preset == CUSTOM_VERSION_PRESET:
        if not custom_version.strip():
            raise ConfigurationError('Для preset custom заполните CUSTOM_BLENDER_VERSION.')
        return validate_blender_version(custom_version)
    if preset not in LTS_VERSION_PRESETS:
        raise ConfigurationError('Выберите LTS preset или custom.')
    if custom_version.strip():
        raise ConfigurationError('CUSTOM_BLENDER_VERSION разрешён только для preset custom.')
    return preset

@dataclass(frozen=True)
class RenderConfig:
    blender_version: str
    enable_cycles_gpu: bool
    allow_cpu_fallback: bool
    include_cpu_with_gpu: bool
    render_mode: RenderMode
    still_frame: int
    download_result: bool
    run_cycles_smoke_test: bool
    run_preflight_test_frame: bool

def validate_config():
    values = {
        'ENABLE_CYCLES_GPU': ENABLE_CYCLES_GPU,
        'ALLOW_CPU_FALLBACK': ALLOW_CPU_FALLBACK,
        'INCLUDE_CPU_WITH_GPU': INCLUDE_CPU_WITH_GPU,
        'DOWNLOAD_RESULT': DOWNLOAD_RESULT,
        'RUN_CYCLES_SMOKE_TEST': RUN_CYCLES_SMOKE_TEST,
        'RUN_PREFLIGHT_TEST_FRAME': RUN_PREFLIGHT_TEST_FRAME,
    }
    for name, value in values.items():
        if not isinstance(value, bool):
            raise ConfigurationError(f'{name} должен быть True или False.')
    if isinstance(STILL_FRAME, bool) or not isinstance(STILL_FRAME, int) or STILL_FRAME < 0:
        raise ConfigurationError('STILL_FRAME должен быть неотрицательным целым числом.')
    try:
        render_mode = RenderMode(RENDER_MODE.strip().upper())
    except (AttributeError, ValueError) as error:
        raise ConfigurationError('RENDER_MODE должен быть STILL или ANIMATION.') from error
    if INCLUDE_CPU_WITH_GPU and not ENABLE_CYCLES_GPU:
        raise ConfigurationError('INCLUDE_CPU_WITH_GPU требует ENABLE_CYCLES_GPU=True.')
    return RenderConfig(
        blender_version=resolve_blender_version(BLENDER_VERSION_PRESET, CUSTOM_BLENDER_VERSION),
        enable_cycles_gpu=ENABLE_CYCLES_GPU,
        allow_cpu_fallback=ALLOW_CPU_FALLBACK,
        include_cpu_with_gpu=INCLUDE_CPU_WITH_GPU,
        render_mode=render_mode,
        still_frame=STILL_FRAME,
        download_result=DOWNLOAD_RESULT,
        run_cycles_smoke_test=RUN_CYCLES_SMOKE_TEST,
        run_preflight_test_frame=RUN_PREFLIGHT_TEST_FRAME,
    )

CONFIG = validate_config()
BLENDER_VERSION = CONFIG.blender_version
print(f'Blender: {BLENDER_VERSION}; Cycles GPU: {CONFIG.enable_cycles_gpu}; CPU fallback: {CONFIG.allow_cpu_fallback}')

In [ ]:
#@title 2. Установка Blender с официальной SHA-256 проверкой
import hashlib
from pathlib import Path
import shutil
import subprocess
import tempfile
from urllib.request import Request, urlopen

blender_series = '.'.join(BLENDER_VERSION.split('.')[:2])
archive_name = f'blender-{BLENDER_VERSION}-linux-x64.tar.xz'
release_url = f'https://download.blender.org/release/Blender{blender_series}'
archive_url = f'{release_url}/{archive_name}'
checksum_url = f'{release_url}/blender-{BLENDER_VERSION}.sha256'
install_root = Path('/content/blender')
blender_dir = install_root / f'blender-{BLENDER_VERSION}-linux-x64'
BLENDER = blender_dir / 'blender'

def checksum_for_archive(manifest, filename):
    for line in manifest.splitlines():
        fields = line.strip().split(maxsplit=1)
        if len(fields) == 2 and fields[1].lstrip('*') == filename:
            if re.fullmatch(r'[0-9a-fA-F]{64}', fields[0]):
                return fields[0].lower()
    raise RuntimeError(f'Официальный checksum manifest не содержит {filename}.')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

OFFICIAL_DOWNLOAD_USER_AGENT = 'Blender-to-GoogleColab/phase-1'

def open_official_url(url):
    return urlopen(Request(url, headers={'User-Agent': OFFICIAL_DOWNLOAD_USER_AGENT}))

def download_verified_archive(destination):
    with open_official_url(checksum_url) as response:
        expected_checksum = checksum_for_archive(response.read().decode('utf-8'), archive_name)
    with tempfile.NamedTemporaryFile(dir=destination.parent, prefix=f'.{archive_name}.', delete=False) as temporary:
        temporary_path = Path(temporary.name)
        try:
            with open_official_url(archive_url) as response:
                shutil.copyfileobj(response, temporary)
            actual_checksum = sha256_file(temporary_path)
            if actual_checksum != expected_checksum:
                raise RuntimeError('SHA-256 Blender archive не совпал; частичная загрузка удалена.')
            temporary_path.replace(destination)
        except Exception:
            temporary_path.unlink(missing_ok=True)
            raise

if not BLENDER.exists():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libgl1', 'libglib2.0-0', 'libsm6'], check=True)
    install_root.mkdir(parents=True, exist_ok=True)
    archive_path = install_root / archive_name
    download_verified_archive(archive_path)
    subprocess.run(['tar', '-xf', str(archive_path), '-C', str(install_root)], check=True)

if not BLENDER.exists():
    raise RuntimeError(f'Blender не найден после распаковки: {BLENDER}')
subprocess.run([str(BLENDER), '--version'], check=True)

In [ ]:
#@title 2.1. Реальный Cycles GPU smoke test (опционально)
# При RUN_CYCLES_SMOKE_TEST=True создаётся новая временная сцена. Загруженный проект не открывается и не изменяется.
if CONFIG.run_cycles_smoke_test:
    smoke_script = Path('/content/cycles_gpu_smoke_test.py')
    smoke_log = Path('/content/cycles_gpu_smoke_test.log')
    smoke_script.write_text('''import bpy
import math

def refresh_devices(preferences):
    refresh = getattr(preferences, 'refresh_devices', None)
    if refresh is not None:
        refresh()
    else:
        preferences.get_devices()

addon = bpy.context.preferences.addons.get('cycles')
if addon is None:
    raise RuntimeError('Cycles add-on/preferences are unavailable.')
preferences = addon.preferences
selected_backend = None
selected_devices = []
errors = []
for backend in ('OPTIX', 'CUDA'):
    try:
        preferences.compute_device_type = backend
        refresh_devices(preferences)
        devices = tuple(preferences.devices)
        selected_devices = [device for device in devices if str(device.type).upper() == backend]
        if not selected_devices:
            errors.append(f'{backend}: no matching Cycles devices')
            continue
        for device in devices:
            device.use = str(device.type).upper() == backend
        selected_backend = backend
        break
    except Exception as error:
        errors.append(f'{backend}: {error}')
if selected_backend is None:
    raise RuntimeError('Cycles GPU smoke test requires GPU; ' + '; '.join(errors))

scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'GPU'
scene.cycles.samples = 1
scene.render.resolution_x = 64
scene.render.resolution_y = 64
scene.render.resolution_percentage = 100
bpy.ops.mesh.primitive_cube_add()
bpy.ops.object.camera_add(location=(0, -3, 0))
camera = bpy.context.object
camera.rotation_euler = (math.radians(90), 0, 0)
scene.camera = camera
bpy.ops.object.light_add(type='POINT', location=(2, -2, 3))
bpy.context.object.data.energy = 1000
bpy.ops.render.render(write_still=False)
names = [f'{device.name} ({device.type})' for device in selected_devices]
print(f'CYCLES_SMOKE_TEST_PASS backend={selected_backend}; devices={names}')
''', encoding='utf-8')
    result = subprocess.run(
        [str(BLENDER), '--background', '--factory-startup', '--python', str(smoke_script)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    smoke_log.write_text(result.stdout, encoding='utf-8')
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'Cycles GPU smoke test failed; log: {smoke_log}')
    if 'CYCLES_SMOKE_TEST_PASS backend=' not in result.stdout:
        raise RuntimeError(f'Cycles smoke test produced no GPU confirmation; log: {smoke_log}')
    print(f'Cycles GPU smoke test confirmed by Blender; log: {smoke_log}')
else:
    print('Cycles GPU smoke test is prepared. Set RUN_CYCLES_SMOKE_TEST=True and rerun this cell in a GPU Colab runtime.')

In [ ]:
#@title 3. Безопасная загрузка `.blend` или ZIP-проекта
from google.colab import files
from pathlib import PurePosixPath, PureWindowsPath
import shutil
import stat
import zipfile

MAX_ARCHIVE_ENTRIES = 10_000
MAX_ARCHIVE_UNCOMPRESSED_BYTES = 25 * 1024 ** 3
MAX_ARCHIVE_COMPRESSION_RATIO = 100

class ArchiveSafetyError(ValueError):
    pass

def validate_zip_member(info):
    name = info.filename
    posix_path = PurePosixPath(name)
    windows_path = PureWindowsPath(name)
    if not name or '\x00' in name:
        raise ArchiveSafetyError('ZIP содержит пустое или NUL-имя entry.')
    if posix_path.is_absolute() or windows_path.is_absolute() or windows_path.drive:
        raise ArchiveSafetyError(f'ZIP содержит абсолютный путь: {name!r}')
    if any(part == '..' for part in posix_path.parts):
        raise ArchiveSafetyError(f'ZIP содержит path traversal: {name!r}')
    mode = info.external_attr >> 16
    file_type = stat.S_IFMT(mode)
    if file_type and not (stat.S_ISREG(mode) or stat.S_ISDIR(mode)):
        raise ArchiveSafetyError(f'ZIP содержит special file или symlink: {name!r}')

def safe_extract_zip(archive_path, destination):
    with zipfile.ZipFile(archive_path) as archive:
        infos = tuple(archive.infolist())
        if len(infos) > MAX_ARCHIVE_ENTRIES:
            raise ArchiveSafetyError(f'Слишком много entries: {len(infos)} > {MAX_ARCHIVE_ENTRIES}.')
        total_uncompressed = 0
        for info in infos:
            validate_zip_member(info)
            if info.is_dir():
                continue
            if info.file_size and info.compress_size == 0:
                raise ArchiveSafetyError(f'Некорректный compressed size: {info.filename!r}')
            if info.file_size / max(info.compress_size, 1) > MAX_ARCHIVE_COMPRESSION_RATIO:
                raise ArchiveSafetyError(f'Подозрительный compression ratio: {info.filename!r}')
            total_uncompressed += info.file_size
            if total_uncompressed > MAX_ARCHIVE_UNCOMPRESSED_BYTES:
                raise ArchiveSafetyError('Распакованный размер ZIP превышает лимит безопасности.')
        destination.mkdir(parents=True, exist_ok=True)
        if destination.is_symlink():
            raise ArchiveSafetyError('Каталог распаковки не может быть symbolic link.')
        if total_uncompressed > shutil.disk_usage(destination).free:
            raise ArchiveSafetyError('Недостаточно свободного места для безопасной распаковки.')
        root = destination.resolve()
        for info in infos:
            target = root.joinpath(*PurePosixPath(info.filename).parts)
            try:
                target.resolve().relative_to(root)
            except ValueError as error:
                raise ArchiveSafetyError(f'ZIP entry выходит за staging: {info.filename!r}') from error
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open('xb') as output:
                shutil.copyfileobj(source, output)

workspace = Path('/content/blender_project')
shutil.rmtree(workspace, ignore_errors=True)
uploads_dir = workspace / 'uploads'
project_dir = workspace / 'project'
uploads_dir.mkdir(parents=True)

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Загрузите ровно один .blend или ZIP-архив проекта.')
name, data = next(iter(uploaded.items()))
uploaded_name = Path(name).name
if not uploaded_name or Path(uploaded_name).suffix.lower() not in {'.blend', '.zip'}:
    raise RuntimeError('Поддерживается только один файл .blend или .zip.')
uploaded_path = uploads_dir / uploaded_name
uploaded_path.write_bytes(data)
if uploaded_path.suffix.lower() == '.zip':
    safe_extract_zip(uploaded_path, project_dir)
else:
    project_dir.mkdir()
    shutil.copy2(uploaded_path, project_dir / uploaded_name)

blend_files = sorted(project_dir.rglob('*.blend'))
if len(blend_files) != 1:
    raise RuntimeError(f'Найдено .blend-файлов: {len(blend_files)}. Загрузите проект с ровно одним .blend.')
BLEND_FILE = blend_files[0]
print(f'Будет отрендерен: {BLEND_FILE}')

In [ ]:
#@title 4. Read-only preflight проекта (JSON)
# Проверка запускает stock Blender с --factory-startup и --disable-autoexec, не вызывает bpy.ops и сверяет SHA-256 исходного .blend.
import hashlib
import json
import time

PREFLIGHT_SCRIPT = r'''import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import bpy

def issue(report, severity, code, message, **details):
    entry = {'code': code, 'message': message}
    if details:
        entry['details'] = details
    report['issues'][severity].append(entry)

def abspath(filepath, library=None):
    if not filepath:
        return ''
    try:
        return bpy.path.abspath(filepath, library=library)
    except TypeError:
        return bpy.path.abspath(filepath)

def add_asset(report, kind, block, filepath=None, packed=False):
    name = block.name
    if packed:
        report['assets'].append({'kind': kind, 'name': name, 'status': 'packed_in_blend', 'path': None})
        return
    path = abspath(filepath if filepath is not None else getattr(block, 'filepath', ''), getattr(block, 'library', None))
    if not path:
        report['assets'].append({'kind': kind, 'name': name, 'status': 'not_applicable', 'path': None})
        return
    exists = os.path.exists(path)
    report['assets'].append({'kind': kind, 'name': name, 'status': 'available' if exists else 'missing', 'path': path})
    if not exists:
        issue(report, 'errors', 'missing_asset', f'Missing {kind}: {name}', path=path)

def gpu_info():
    command = ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits']
    try:
        result = subprocess.run(command, check=False, capture_output=True, text=True, timeout=10)
    except (OSError, subprocess.SubprocessError) as error:
        return {'available': False, 'devices': [], 'error': str(error)}
    if result.returncode:
        return {'available': False, 'devices': [], 'error': result.stderr.strip()}
    devices = []
    for line in result.stdout.splitlines():
        fields = [field.strip() for field in line.split(',')]
        if len(fields) == 3:
            devices.append({'name': fields[0], 'vram_total_mib': fields[1], 'vram_free_mib': fields[2]})
    return {'available': bool(devices), 'devices': devices}

def cycles_devices():
    addon = bpy.context.preferences.addons.get('cycles')
    if addon is None:
        return {'available': False, 'devices': []}
    try:
        preferences = addon.preferences
        refresh = getattr(preferences, 'refresh_devices', None)
        refresh() if refresh else preferences.get_devices()
        return {'available': True, 'devices': [{'name': device.name, 'type': str(device.type), 'enabled': bool(device.use)} for device in preferences.devices]}
    except Exception as error:
        return {'available': True, 'devices': [], 'error': str(error)}

def build_report(report_path):
    scenes = tuple(bpy.data.scenes)
    context_scene_name = getattr(getattr(bpy.context, 'scene', None), 'name', '')
    active_scene_name = context_scene_name if context_scene_name in {scene.name for scene in scenes} else scenes[0].name
    report = {'report_version': 1, 'source_file': bpy.data.filepath, 'blender': {'version': bpy.app.version_string, 'version_file': list(bpy.data.version)}, 'active_scene': active_scene_name, 'scenes': [], 'assets': [], 'issues': {'errors': [], 'warnings': []}, 'environment': {}}
    for scene in bpy.data.scenes:
        render = scene.render
        record = {'name': scene.name, 'camera': scene.camera.name if scene.camera else None, 'render_engine': render.engine, 'frame_start': scene.frame_start, 'frame_end': scene.frame_end, 'frame_step': scene.frame_step, 'fps': render.fps / render.fps_base, 'resolution': {'x': render.resolution_x, 'y': render.resolution_y, 'percentage': render.resolution_percentage}, 'output': {'filepath': abspath(render.filepath), 'file_format': render.image_settings.file_format, 'color_mode': render.image_settings.color_mode}}
        report['scenes'].append(record)
        if not record['camera']:
            issue(report, 'errors', 'missing_camera', f'Scene {scene.name!r} has no active camera.')
        if record['render_engine'] not in {'CYCLES', 'BLENDER_EEVEE', 'BLENDER_EEVEE_NEXT', 'BLENDER_WORKBENCH'}:
            issue(report, 'errors', 'unsupported_render_engine', f'Scene {scene.name!r} uses {record["render_engine"]!r}.')
        if record['frame_end'] < record['frame_start'] or record['frame_step'] < 1:
            issue(report, 'errors', 'invalid_frame_range', f'Scene {scene.name!r} has an invalid frame range.')
    for image in bpy.data.images:
        if image.source not in {'GENERATED', 'VIEWER'}:
            add_asset(report, 'image', image, packed=bool(getattr(image, 'packed_file', None)) or bool(getattr(image, 'packed_files', ())))
    for collection, kind in ((bpy.data.libraries, 'linked_library'), (bpy.data.fonts, 'font'), (bpy.data.volumes, 'vdb'), (bpy.data.cache_files, 'cache'), (bpy.data.movieclips, 'movie_clip'), (bpy.data.sounds, 'sound')):
        for block in collection:
            add_asset(report, kind, block)
    for obj in bpy.data.objects:
        for modifier in obj.modifiers:
            domain = getattr(modifier, 'domain_settings', None)
            cache_directory = getattr(domain, 'cache_directory', '')
            if cache_directory:
                add_asset(report, 'fluid_cache', obj, cache_directory)
    try:
        ram_total_bytes = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
    except (AttributeError, ValueError, OSError):
        ram_total_bytes = None
    disk = shutil.disk_usage(report_path.parent)
    system_root = Path(bpy.app.binary_path).resolve().parent
    external_addons = []
    for addon in bpy.context.preferences.addons:
        if addon.module == 'cycles':
            continue
        module_path = getattr(sys.modules.get(addon.module), '__file__', None)
        if not module_path:
            continue
        try:
            Path(module_path).resolve().relative_to(system_root)
            continue
        except ValueError:
            pass
        external_addons.append(addon.module)
    external_addons.sort()
    report['environment'] = {'ram_total_bytes': ram_total_bytes, 'disk_total_bytes': disk.total, 'disk_free_bytes': disk.free, 'gpu': gpu_info(), 'cycles_device_discovery': cycles_devices(), 'enabled_external_addons': external_addons}
    if external_addons:
        issue(report, 'warnings', 'external_addons_enabled', 'Enabled add-ons may be unavailable in stock Colab Blender.', modules=external_addons)
    return report

separator = sys.argv.index('--')
report_path = Path(sys.argv[separator + 1])
try:
    report = build_report(report_path)
except Exception as error:
    report = {'report_version': 1, 'source_file': bpy.data.filepath, 'blender': {'version': bpy.app.version_string}, 'active_scene': None, 'scenes': [], 'assets': [], 'issues': {'errors': [{'code': 'probe_failure', 'message': str(error)}], 'warnings': []}, 'environment': {}}
report_path.parent.mkdir(parents=True, exist_ok=True)
temporary_path = report_path.with_suffix(report_path.suffix + '.tmp')
temporary_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True), encoding='utf-8')
temporary_path.replace(report_path)
print(f'BLENDER_PREFLIGHT_REPORT={report_path}')
'''

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

preflight_dir = Path('/content/blender_preflight') / BLEND_FILE.stem
preflight_dir.mkdir(parents=True, exist_ok=True)
probe_script = preflight_dir / 'blender_preflight.py'
preflight_report_path = preflight_dir / 'preflight.json'
probe_script.write_text(PREFLIGHT_SCRIPT, encoding='utf-8')
source_hash_before_probe = sha256_file(BLEND_FILE)
probe_command = [str(BLENDER), '--background', '--factory-startup', '--disable-autoexec', str(BLEND_FILE), '--python', str(probe_script), '--', str(preflight_report_path)]
probe_result = subprocess.run(probe_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
source_hash_after_probe = sha256_file(BLEND_FILE)
if source_hash_after_probe != source_hash_before_probe:
    raise RuntimeError('Read-only preflight изменил исходный .blend; дальнейший рендер заблокирован.')
if probe_result.returncode != 0 or not preflight_report_path.exists():
    raise RuntimeError(f'Blender preflight не завершился: {probe_result.stdout}')
PREFLIGHT_REPORT = json.loads(preflight_report_path.read_text(encoding='utf-8'))
required_keys = {'report_version', 'source_file', 'blender', 'scenes', 'assets', 'issues', 'environment'}
if PREFLIGHT_REPORT.get('report_version') != 1 or required_keys.difference(PREFLIGHT_REPORT):
    raise RuntimeError('Blender preflight вернул несовместимый JSON report.')
errors = PREFLIGHT_REPORT['issues']['errors']
warnings = PREFLIGHT_REPORT['issues']['warnings']
print(json.dumps(PREFLIGHT_REPORT, ensure_ascii=False, indent=2))
print(f'Preflight: {len(errors)} error(s), {len(warnings)} warning(s); SHA-256 source сохранён.')

if CONFIG.run_preflight_test_frame:
    if errors:
        raise RuntimeError('Test-frame заблокирован ошибками preflight; исправьте их до расходования GPU-времени.')
    active_scene = next(scene for scene in PREFLIGHT_REPORT['scenes'] if scene['name'] == PREFLIGHT_REPORT['active_scene'])
    test_frame = active_scene['frame_start']
    test_dir = preflight_dir / f'test_frame_{int(time.time())}'
    test_dir.mkdir()
    test_prefix = test_dir / 'frame_'
    print(f'Runtime-only test override: frame={test_frame}; output={test_prefix}; source .blend не сохраняется.')
    test_command = [str(BLENDER), '--background', '--factory-startup', '--disable-autoexec', str(BLEND_FILE)]
    if CONFIG.enable_cycles_gpu:
        test_gpu_script = test_dir / 'configure_cycles_gpu.py'
        test_gpu_script.write_text(f'''import bpy
ALLOW_CPU_FALLBACK = {CONFIG.allow_cpu_fallback!r}
INCLUDE_CPU_WITH_GPU = {CONFIG.include_cpu_with_gpu!r}
addon = bpy.context.preferences.addons.get('cycles')
cycles_scenes = [scene for scene in bpy.data.scenes if scene.render.engine == 'CYCLES']
if cycles_scenes:
    if addon is None:
        raise RuntimeError('Cycles preferences are unavailable.')
    preferences = addon.preferences
    failures = []
    for backend in ('OPTIX', 'CUDA'):
        try:
            preferences.compute_device_type = backend
            refresh = getattr(preferences, 'refresh_devices', None)
            refresh() if refresh else preferences.get_devices()
            devices = tuple(preferences.devices)
            selected = [device for device in devices if str(device.type).upper() == backend]
            if not selected:
                failures.append(f'{{backend}}: no matching Cycles devices')
                continue
            for device in devices:
                device.use = str(device.type).upper() == backend or (INCLUDE_CPU_WITH_GPU and str(device.type).upper() == 'CPU')
            for scene in cycles_scenes:
                scene.cycles.device = 'GPU'
            print(f'CYCLES_DEVICE: backend={{backend}}')
            break
        except Exception as error:
            failures.append(f'{{backend}}: {{error}}')
    else:
        if not ALLOW_CPU_FALLBACK:
            raise RuntimeError('Cycles GPU was not configured. CPU fallback is disabled. ' + '; '.join(failures))
        for scene in cycles_scenes:
            scene.cycles.device = 'CPU'
''', encoding='utf-8')
        test_command += ['--python', str(test_gpu_script)]
    test_command += ['-o', str(test_prefix), '-f', str(test_frame)]
    source_hash_before_test = sha256_file(BLEND_FILE)
    started_at = time.monotonic()
    try:
        subprocess.run(test_command, check=True)
    finally:
        if sha256_file(BLEND_FILE) != source_hash_before_test:
            raise RuntimeError('Test-frame изменил исходный .blend; дальнейший рендер заблокирован.')
    elapsed_seconds = time.monotonic() - started_at
    test_output_bytes = sum(path.stat().st_size for path in test_dir.rglob('*') if path.is_file())
    frame_count = ((active_scene['frame_end'] - active_scene['frame_start']) // active_scene['frame_step']) + 1
    estimate = {'frame_count': frame_count, 'test_frame_seconds': elapsed_seconds, 'test_frame_output_bytes': test_output_bytes, 'estimated_render_seconds': elapsed_seconds * frame_count, 'estimated_output_bytes': test_output_bytes * frame_count, 'note': 'Линейная оценка по одному кадру; сложность кадров и startup Blender могут отличаться.'}
    (test_dir / 'estimate.json').write_text(json.dumps(estimate, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(estimate, ensure_ascii=False, indent=2))
else:
    print('Test-frame не запускался. Установите RUN_PREFLIGHT_TEST_FRAME=True только после просмотра JSON report.')

In [ ]:
# Подключите Google Drive и задайте папку для результатов.
# Colab покажет ссылку для авторизации доступа к вашему Google Drive.
from google.colab import drive
from datetime import datetime

drive.mount('/content/drive')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/Blender Renders')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Рендеры будут сохранены в Google Drive: {DRIVE_OUTPUT_DIR}')

In [ ]:
#@title 5. Выбор устройства Cycles и запуск рендера
# Скрипт применяется только к процессу Blender; исходный .blend не сохраняется.
gpu_script = Path('/content/configure_cycles_gpu.py')
gpu_script.write_text(f'''import bpy

ALLOW_CPU_FALLBACK = {CONFIG.allow_cpu_fallback!r}
INCLUDE_CPU_WITH_GPU = {CONFIG.include_cpu_with_gpu!r}

def refresh_devices(preferences):
    refresh = getattr(preferences, 'refresh_devices', None)
    if refresh is not None:
        refresh()
        return
    legacy_get_devices = getattr(preferences, 'get_devices', None)
    if legacy_get_devices is None:
        raise RuntimeError('Cycles preferences do not expose device discovery.')
    legacy_get_devices()

def configure_cycles():
    cycles_scenes = [scene for scene in bpy.data.scenes if scene.render.engine == 'CYCLES']
    if not cycles_scenes:
        print('CYCLES_DEVICE: no Cycles scenes; no device selection required.')
        return
    addon = bpy.context.preferences.addons.get('cycles')
    if addon is None:
        raise RuntimeError('Cycles add-on/preferences are unavailable.')
    preferences = addon.preferences
    failures = []
    for backend in ('OPTIX', 'CUDA'):
        try:
            preferences.compute_device_type = backend
            refresh_devices(preferences)
            devices = tuple(preferences.devices)
            selected_gpus = [device for device in devices if str(device.type).upper() == backend]
            if not selected_gpus:
                failures.append(f'{{backend}}: no matching Cycles devices')
                continue
            for device in devices:
                device_type = str(device.type).upper()
                device.use = device_type == backend or (INCLUDE_CPU_WITH_GPU and device_type == 'CPU')
            for scene in cycles_scenes:
                scene.cycles.device = 'GPU'
            active = [f'{{device.name}} ({{device.type}})' for device in devices if device.use]
            print(f'CYCLES_DEVICE: backend={{backend}}; active={{active}}')
            return
        except Exception as error:
            failures.append(f'{{backend}}: {{error}}')
    if ALLOW_CPU_FALLBACK:
        for scene in cycles_scenes:
            scene.cycles.device = 'CPU'
        print('CYCLES_DEVICE: CPU fallback explicitly allowed; ' + '; '.join(failures))
        return
    raise RuntimeError('Cycles GPU was not configured. CPU fallback is disabled. ' + '; '.join(failures))

configure_cycles()
''', encoding='utf-8')

# Отдельная папка на каждый запуск не даёт перезаписать прошлый рендер.
render_name = f'{BLEND_FILE.stem}_{datetime.now():%Y-%m-%d_%H-%M-%S}'
output_dir = DRIVE_OUTPUT_DIR / render_name
output_dir.mkdir()
output_prefix = output_dir / 'frame_'

command = [str(BLENDER), '-b', str(BLEND_FILE)]
if CONFIG.enable_cycles_gpu:
    command += ['-P', str(gpu_script)]
command += ['-o', str(output_prefix)]
if CONFIG.render_mode is RenderMode.STILL:
    command += ['-f', str(CONFIG.still_frame)]
elif CONFIG.render_mode is RenderMode.ANIMATION:
    command += ['-a']
else:
    raise ValueError("RENDER_MODE должен быть 'STILL' или 'ANIMATION'.")

print('Аргументы процесса Blender:', command)
subprocess.run(command, check=True)

In [ ]:
# Результат уже сохранён в Google Drive. При DOWNLOAD_RESULT = True
# эта ячейка дополнительно скачает файл/ZIP на компьютер.
rendered_files = [path for path in output_dir.rglob('*') if path.is_file()]
if not rendered_files:
    raise RuntimeError('Рендер завершился, но файлов в папке результата нет.')

print(f'Готово: {len(rendered_files)} файл(ов) в Google Drive: {output_dir}')

if CONFIG.download_result:
    if len(rendered_files) == 1:
        result = rendered_files[0]
    else:
        result_base = DRIVE_OUTPUT_DIR / output_dir.name
        shutil.make_archive(str(result_base), 'zip', output_dir)
        result = result_base.with_suffix('.zip')
    print(f'Скачивается: {result.name} ({result.stat().st_size / 1024 / 1024:.1f} MB)')
    files.download(str(result))